In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────
!pip install yfinance pandas numpy backtesting -q

In [ ]:
# ── Cell 2: Imports & Config ──────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from backtesting import Backtest, Strategy

# ── Symbols ───────────────────────────────────────────────────────────
# yfinance uses .CA suffix for Cairo (EGX) listed stocks.
# Add more symbols here as needed — each needs a MANUAL_LEVELS entry.
SYMBOLS = {
    'COMI': 'COMI.CA',
    'ETEL': 'ETEL.CA',
}

# ── Date range ────────────────────────────────────────────────────────
FROM_DATE = '2024-01-01'
TO_DATE   = datetime.today().strftime('%Y-%m-%d')

# ── Manual Support / Resistance (your input only) ─────────────────────
# This is the ONLY source of S/R levels in the system.
# If a symbol is missing here, it will be skipped with a clear warning.
# No dynamic detection. No fallback. Your levels only.
#
# GUIDELINE: The zone between support and resistance should be wider
# than 2x the stock's daily ATR. You will see the ATR in the output.
# If the zone is too narrow, signals around S/R become noise.
MANUAL_LEVELS = {
    'COMI': {'support': 122.00, 'resistance': 132.00},
    'ETEL': {'support': 80.00,  'resistance': 87.00},
}

# ── User risk profile ─────────────────────────────────────────────────
# This will come from user input in the final system.
# Controls signal thresholds and capital risk per trade.
#
# 'conservative' : needs score >= 4 to BUY, risks 1% per trade
# 'moderate'     : needs score >= 2 to BUY, risks 2% per trade
# 'aggressive'   : needs score >= 1 to BUY, risks 3% per trade
USER_RISK_PROFILE = 'moderate'
USER_CAPITAL      = 10000.0

RISK_PROFILES = {
    'conservative': {
        'buy_threshold':         4,
        'strong_buy_threshold':  7,
        'sell_threshold':       -4,
        'strong_sell_threshold':-7,
        'risk_pct_per_trade':    0.01,
        'description': 'Conservative — only acts on strong confirmed signals'
    },
    'moderate': {
        'buy_threshold':         2,
        'strong_buy_threshold':  5,
        'sell_threshold':       -2,
        'strong_sell_threshold':-5,
        'risk_pct_per_trade':    0.02,
        'description': 'Moderate — balanced signal quality and opportunity'
    },
    'aggressive': {
        'buy_threshold':         1,
        'strong_buy_threshold':  3,
        'sell_threshold':       -1,
        'strong_sell_threshold':-3,
        'risk_pct_per_trade':    0.03,
        'description': 'Aggressive — acts on early signals, accepts more noise'
    },
}

# ── Transaction costs ─────────────────────────────────────────────────
# EGX realistic estimate: brokerage + stamp duty + FRA fees ≈ 0.8% round trip.
# backtesting.py applies this as a per-trade commission automatically.
TRANSACTION_COST_PCT = 0.008

# ── REFORM #1: Slippage & Volume Filter ──────────────────────────────
# EGX stocks are often thinly traded. Executing at the prior-bar close
# price is optimistic. We add:
#
#   SLIPPAGE_PCT   — price paid is worse by this % on every entry/exit.
#                    0.3% is conservative for mid-cap EGX names.
#                    Raise to 0.5% for small-caps.
#
#   MIN_DAILY_VOLUME_EGP — minimum daily traded value (EGP) to even
#                    consider a trade. Prevents signals on days when
#                    you literally cannot fill an order at any price.
#                    Default 500,000 EGP ≈ $10k USD — very conservative.
#
# These values are applied INSIDE the backtest and also checked in
# the live signal (MIN_DAILY_VOLUME_EGP only — slippage is backtest-only).
SLIPPAGE_PCT          = 0.003   # 0.3% per entry and per exit
MIN_DAILY_VOLUME_EGP  = 500_000 # Skip trade if volume × price < this

# ── REFORM #2: Two-Layer Entry — Setup vs Confirmation ───────────────
# The old system used the same threshold for BOTH detecting a setup
# and confirming entry. This produced very few trades (3–5 over 2 years)
# making backtests statistically meaningless.
#
# New architecture:
#   SETUP layer   — looser threshold (score ≥ 1 for moderate).
#                   Detects that a trade is worth watching.
#                   Used in backtesting to generate enough trades.
#   CONFIRM layer — user's risk profile threshold (score ≥ 2 for moderate).
#                   Filters setups into actionable live signals.
#
# In backtesting we use SETUP_SCORE_OFFSET to loosen entry by 1 point.
# This gives the backtest realistic trade frequency while the live
# signal stays at the user's chosen threshold.
SETUP_SCORE_OFFSET = 1  # Backtest enters at profile threshold − this

# ── Backtest reliability threshold ───────────────────────────────────
# Results with fewer trades than this are flagged as statistically weak.
# With fewer than 8 trades, win rate and return figures are meaningless.
MIN_TRADES_FOR_RELIABLE_BACKTEST = 8

# ── Local cache ───────────────────────────────────────────────────────
CACHE_DIR = 'egx_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

profile = RISK_PROFILES[USER_RISK_PROFILE]
print('✅ Config ready')
print(f'   Period       : {FROM_DATE} → {TO_DATE}')
print(f'   Symbols      : {list(SYMBOLS.keys())}')
print(f'   Risk profile : {USER_RISK_PROFILE} — {profile["description"]}')
print(f'   Capital      : EGP {USER_CAPITAL:,.0f}')
print(f'   Tx cost      : {TRANSACTION_COST_PCT*100}% round trip')
print(f'   Slippage     : {SLIPPAGE_PCT*100}% per side (backtest realism)')
print(f'   Min volume   : EGP {MIN_DAILY_VOLUME_EGP:,.0f}/day to qualify')
print(f'   Setup offset : -{SETUP_SCORE_OFFSET} vs live threshold (backtest entry)')


In [ ]:
# ── Cell 3: Data Fetcher with Local Cache ─────────────────────────────
def fetch_data(symbol):
    """
    Loads from local CSV cache if available.
    Downloads from Yahoo Finance on first run, then saves to cache.
    Delete the CSV file to force a fresh download.

    Returns a clean OHLCV dataframe with DatetimeIndex.
    Returns None if data is unavailable or empty.
    """
    ticker     = SYMBOLS[symbol]
    cache_path = os.path.join(CACHE_DIR, f'{symbol}_{FROM_DATE}_{TO_DATE}.csv')

    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        print(f'  📂 {symbol}: loaded from cache ({len(df)} rows)')
        return df

    print(f'  🌐 {symbol}: downloading from Yahoo Finance...')
    raw = yf.download(ticker, start=FROM_DATE, end=TO_DATE,
                      progress=False, auto_adjust=True)

    if raw.empty:
        print(f'  ❌ {symbol}: no data returned — check ticker "{ticker}"')
        return None

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.index.name = 'date'
    df.to_csv(cache_path)
    print(f'  ✅ {symbol}: downloaded and cached ({len(df)} rows)')
    return df

print('✅ Fetcher ready')

In [ ]:
# ── Cell 4: Safe Accessor ─────────────────────────────────────────────
def safe_iloc(series, pos):
    """
    Returns series.iloc[pos] without raising IndexError.
    Returns NaN if the position does not exist.
    Used everywhere a positional access could fail on short dataframes.
    """
    try:
        return float(series.iloc[pos])
    except (IndexError, TypeError, ValueError):
        return float('nan')

print('✅ Safe accessor ready')

In [ ]:
# ── Cell 5: Technical Indicators ──────────────────────────────────────
def compute_all_indicators(df):
    """
    Computes all technical indicators and appends them as columns.

    REFORM #3 — Orthogonal Indicators:
    The original system used RSI + MACD + Bollinger, all of which
    measure the same underlying price momentum. This creates "fake
    confirmation" — three signals agreeing because they share inputs,
    not because they capture independent market dimensions.

    New indicator set maps to four independent market dimensions:
      1. MOMENTUM  → RSI (price speed — are buyers/sellers exhausted?)
      2. TREND STR → ADX (how strong is the prevailing trend direction?)
      3. STRUCTURE → Bollinger (where is price within its normal range?)
      4. VOLUME    → OBV + Vol spike (is money flow confirming price?)

    MACD is retained as a secondary momentum indicator for the LLM
    context / human-readable output, but it is NO LONGER a scored
    sub-signal. It has been replaced by ADX in the scoring system.

    IMPORTANT: Does NOT call dropna() — all original rows are kept.
    This is required by backtesting.py: the indicator dataframe must
    have the same index as the OHLCV data passed to Backtest().
    NaN values in early rows are handled inside next() via isnan checks.
    """
    df = df.copy()

    # ── RSI (14) — DIMENSION: Momentum ───────────────────────────────
    # Measures speed of price movement.
    # <30 = oversold (buyers likely to step in)
    # >70 = overbought (sellers likely to push back)
    # Period 14 is the universal standard for daily charts.
    delta     = df['Close'].diff()
    gain      = delta.clip(lower=0).rolling(14).mean()
    loss      = -delta.clip(upper=0).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))

    # ── MACD (12 / 26 / 9) — context only, NOT scored ────────────────
    # Retained for the LLM summary and human-readable output.
    # No longer contributes to the scoring system (replaced by ADX).
    # Why: MACD and RSI are both price-momentum metrics derived from
    # the same Close series — including both in scoring is redundant.
    exp12             = df['Close'].ewm(span=12, adjust=False).mean()
    exp26             = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']        = exp12 - exp26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

    # ── ADX (14) — DIMENSION: Trend Strength ─────────────────────────
    # Wilder's Average Directional Index.
    # Measures HOW STRONG the current trend is — not its direction.
    # ADX > 25 = trending (directional trade valid)
    # ADX < 20 = ranging (mean-reversion logic more appropriate)
    # Combined with +DI / -DI to confirm direction:
    #   +DI > -DI = upward directional pressure
    #   -DI > +DI = downward directional pressure
    #
    # WHY THIS REPLACES MACD IN SCORING:
    # ADX captures a truly independent dimension — trend existence and
    # strength — that none of RSI, Bollinger, or OBV measures.
    # This removes the fake confirmation problem where 3 momentum
    # metrics agree simply because they all read the same price move.
    high, low, close = df['High'], df['Low'], df['Close']

    plus_dm  = high.diff()
    minus_dm = -low.diff()
    plus_dm[plus_dm   < 0] = 0
    minus_dm[minus_dm < 0] = 0
    plus_dm[(plus_dm  <= minus_dm)] = 0
    minus_dm[(minus_dm < plus_dm)]  = 0

    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)

    atr14          = tr.rolling(14).mean()
    df['ADX_Plus']  = (plus_dm.rolling(14).mean()  / atr14 * 100)
    df['ADX_Minus'] = (minus_dm.rolling(14).mean() / atr14 * 100)

    # Smooth the directional index difference to get ADX
    dx = ((df['ADX_Plus'] - df['ADX_Minus']).abs() /
          (df['ADX_Plus'] + df['ADX_Minus']).replace(0, np.nan) * 100)
    df['ADX'] = dx.ewm(span=14, adjust=False).mean()

    # ── Bollinger Bands (20, 2 standard deviations) ───────────────────
    # DIMENSION: Price Structure
    # Dynamic price channel around the 20-day moving average.
    # BB_Position: 0.0 = at lower band, 1.0 = at upper band.
    # BB_Width: measures volatility. Narrow = consolidation, Wide = trending.
    df['BB_Mid']      = df['Close'].rolling(20).mean()
    std20             = df['Close'].rolling(20).std()
    df['BB_Upper']    = df['BB_Mid'] + 2 * std20
    df['BB_Lower']    = df['BB_Mid'] - 2 * std20
    df['BB_Width']    = (df['BB_Upper'] - df['BB_Lower']) / df['BB_Mid']
    df['BB_Position'] = (
        (df['Close'] - df['BB_Lower']) /
        (df['BB_Upper'] - df['BB_Lower'])
    )

    # ── ATR (14) — Average True Range ────────────────────────────────
    # Measures average daily price movement (volatility).
    # Used to describe risk to the user — not in the scoring directly.
    hl        = high - low
    hpc       = abs(high - close.shift(1))
    lpc       = abs(low  - close.shift(1))
    df['ATR'] = (
        pd.concat([hl, hpc, lpc], axis=1).max(axis=1).rolling(14).mean()
    )

    # ── OBV — On-Balance Volume ───────────────────────────────────────
    # DIMENSION: Volume / Money Flow
    # Tracks whether volume is confirming or contradicting price.
    # Rising OBV with rising price = healthy move.
    # Falling OBV with rising price = warning, move may not last.
    direction  = np.sign(df['Close'].diff())
    df['OBV']  = (direction * df['Volume']).cumsum()

    # OBV vs its 10-day MA — more stable than raw OBV momentum.
    df['OBV_MA10'] = df['OBV'].rolling(10).mean()

    # ── Volume ────────────────────────────────────────────────────────
    # 20-day average volume — used to detect spikes and confirm moves.
    df['Vol_MA20'] = df['Volume'].rolling(20).mean()

    # ── Moving Averages ───────────────────────────────────────────────
    # SMA20 = short-term trend direction.
    # SMA50 = medium-term trend direction.
    # min_periods=10 on SMA50: produces a value once 10 rows exist
    # rather than waiting for 50 rows, preventing NaN propagation.
    df['SMA20'] = df['Close'].rolling(20).mean()
    df['SMA50'] = df['Close'].rolling(50, min_periods=10).mean()
    df['EMA20'] = df['Close'].ewm(span=20, adjust=False).mean()

    return df

print('✅ Indicators ready (RSI · ADX · Bollinger · OBV/Volume — 4 orthogonal dimensions)')
print('   MACD retained for LLM context only — no longer a scored sub-signal')


In [ ]:
# ── Cell 6: Support & Resistance (manual only) ────────────────────────
def get_levels(symbol):
    """
    Returns (support, resistance) from MANUAL_LEVELS.
    Returns (None, None) if the symbol is not configured.
    No dynamic detection. No fallback. Your levels exclusively.
    """
    if symbol not in MANUAL_LEVELS:
        return None, None
    return MANUAL_LEVELS[symbol]['support'], MANUAL_LEVELS[symbol]['resistance']


def near_and_bouncing(df, level, tolerance=0.02):
    """
    Returns True if price was near the level yesterday AND
    is now moving away from it (basic bounce confirmation).

    WHY: Simply being near a level is not a signal.
    A bounce — price touching and then closing away from the level —
    is a much stronger indication that the level is holding.
    """
    prev_close = safe_iloc(df['Close'], -2)
    curr_close = safe_iloc(df['Close'], -1)
    if np.isnan(prev_close) or np.isnan(curr_close):
        return False
    near   = abs(prev_close - level) / level < tolerance
    bounce = curr_close > prev_close
    return near and bounce

print('✅ Support / Resistance ready (manual only)')

In [ ]:
# ── Cell 7: Trend Detection ───────────────────────────────────────────
def detect_trend(df):
    """
    Classifies trend using two methods combined:

    1. Price structure (Dow Theory): higher highs + higher lows = uptrend.
    2. MA alignment: price > SMA20 > SMA50 confirms uptrend.

    Regime gate: if Bollinger Width < 0.05 (very tight consolidation),
    returns CONSOLIDATING regardless of structure.
    This prevents the trend sub-score from dominating in ranging markets
    where trend signals are inherently unreliable.
    """
    if len(df) < 20:
        return 'UNKNOWN'

    h     = df['High'].tail(20)
    l     = df['Low'].tail(20)
    price = safe_iloc(df['Close'],    -1)
    ma20  = safe_iloc(df['SMA20'],    -1)
    ma50  = safe_iloc(df['SMA50'],    -1)
    bbw   = safe_iloc(df['BB_Width'], -1)

    if np.isnan(ma20) or np.isnan(price):
        return 'UNKNOWN'

    if not np.isnan(bbw) and bbw < 0.05:
        return 'CONSOLIDATING'

    higher_highs = h.iloc[-1] > h.iloc[0]
    higher_lows  = l.iloc[-1] > l.iloc[0]
    lower_highs  = h.iloc[-1] < h.iloc[0]
    lower_lows   = l.iloc[-1] < l.iloc[0]

    ma_up   = price > ma20 and (np.isnan(ma50) or ma20 > ma50)
    ma_down = price < ma20 and (np.isnan(ma50) or ma20 < ma50)

    if higher_highs and higher_lows and ma_up:
        return 'UPTREND'
    elif lower_highs and lower_lows and ma_down:
        return 'DOWNTREND'
    else:
        return 'SIDEWAYS'

print('✅ Trend detection ready')

In [ ]:
# ── Cell 8: Confidence Metric ─────────────────────────────────────────
# ── REFORM #4: Score-Band Confidence (replaces indicator-agreement %) ─
#
# OLD APPROACH — PROBLEM:
#   confidence = (indicators agreeing / total indicators) × 100
#   This was misleading. It counted RSI, Bollinger, and OBV all
#   agreeing, but those three can all agree simply because they
#   react to the same price move — not independent evidence.
#   A 60% "agreement" score was not linked to actual win probability.
#
# NEW APPROACH:
#   confidence = historical win rate for trades at this score band.
#   Each score band (strong buy, buy, sell, strong sell) accumulates
#   a record of past backtest trades and their outcomes.
#   The confidence is drawn from that record.
#
#   If no historical data exists for a band yet, we fall back to a
#   calibrated prior based on signal strength:
#     STRONG BUY/SELL → 65% (strong signal, but unproven)
#     BUY/SELL        → 50% (coin flip until history proves otherwise)
#     HOLD            → 0%  (no directional claim)
#
#   This makes confidence HONEST: it tells the user what they can
#   actually expect from this signal based on evidence, not aesthetics.
#
# SCORE_BAND_HISTORY is populated by the backtest runner (Cell 11)
# after each run. It persists across the session but resets on restart.
# In production, this would be persisted to disk / database.

SCORE_BAND_HISTORY = {
    # Format: 'STRONG BUY': {'wins': 0, 'total': 0}
    'STRONG BUY':  {'wins': 0, 'total': 0},
    'BUY':         {'wins': 0, 'total': 0},
    'SELL':        {'wins': 0, 'total': 0},
    'STRONG SELL': {'wins': 0, 'total': 0},
}

# Calibrated priors — used when history is thin (< MIN_HISTORY_TRADES)
CONFIDENCE_PRIORS = {
    'STRONG BUY':  65.0,
    'BUY':         50.0,
    'SELL':        50.0,
    'STRONG SELL': 65.0,
    'HOLD':         0.0,
}
MIN_HISTORY_TRADES = 5  # Minimum trades before history overrides prior


def update_score_band_history(band, won):
    """
    Called by the backtest runner after each completed trade.
    win = True if the trade was profitable (return > 0).
    """
    if band not in SCORE_BAND_HISTORY:
        return
    SCORE_BAND_HISTORY[band]['total'] += 1
    if won:
        SCORE_BAND_HISTORY[band]['wins'] += 1


def compute_confidence(sub_scores, decision):
    """
    Returns the confidence percentage for a given signal decision.

    Uses historical win rate for the signal's score band if enough
    trades exist. Falls back to calibrated prior otherwise.

    Returns 0.0 for HOLD — no directional claim is made.
    """
    if decision == 'HOLD':
        return 0.0

    history = SCORE_BAND_HISTORY.get(decision, {'wins': 0, 'total': 0})
    total   = history['total']

    if total >= MIN_HISTORY_TRADES:
        win_rate = history['wins'] / total * 100
        return round(win_rate, 1)
    else:
        # Fall back to calibrated prior, noting low sample size
        return CONFIDENCE_PRIORS.get(decision, 50.0)


def get_confidence_note(decision):
    """
    Returns a plain-language note explaining the confidence source.
    Included in JSON output so the LLM can communicate it clearly.
    """
    if decision == 'HOLD':
        return 'No directional signal — confidence not applicable.'
    history = SCORE_BAND_HISTORY.get(decision, {'wins': 0, 'total': 0})
    total   = history['total']
    if total >= MIN_HISTORY_TRADES:
        wins = history['wins']
        return (
            f'Based on {total} historical {decision} signals: '
            f'{wins} winners, {total - wins} losers '
            f'({round(wins/total*100,1)}% win rate).'
        )
    else:
        prior = CONFIDENCE_PRIORS.get(decision, 50.0)
        return (
            f'Calibrated prior ({prior}%) — only {total} historical '
            f'{decision} trade(s) recorded so far '
            f'(need {MIN_HISTORY_TRADES} to use live win rate). '
            f'Run more backtests to build a reliable estimate.'
        )

print('✅ Confidence metric ready (score-band win rate, calibrated prior fallback)')
print('   SCORE_BAND_HISTORY will populate as backtests run')


In [ ]:
# ── Cell 9: Scoring System ────────────────────────────────────────────
def score_symbol(df, support, resistance, risk_profile_name):
    """
    Produces a composite score from 7 sub-signals (-2 to +2 each).
    Maximum possible score: +14 (all signals strongly bullish).
    Minimum possible score: -14 (all signals strongly bearish).

    REFORM #3 — Orthogonal Sub-Signals:
    MACD replaced by ADX as a scored sub-signal.
    The 7 sub-signals now map to 4 independent market dimensions:

      Dimension 1 — MOMENTUM  : RSI
      Dimension 2 — TREND STR : ADX  ← was MACD (correlated with RSI)
      Dimension 3 — STRUCTURE : Bollinger
      Dimension 4 — VOLUME    : Volume spike + OBV  (2 sub-signals)
      Context     — S/R + Trend direction            (2 sub-signals)

    ADX scores differently from MACD:
      +DI > -DI and ADX > 25 = strong uptrend   → +2
      +DI > -DI and ADX > 15 = mild uptrend      → +1
      -DI > +DI and ADX > 25 = strong downtrend  → -2
      -DI > +DI and ADX > 15 = mild downtrend    → -1
      ADX < 15 (no trend)                         →  0
    This is ORTHOGONAL to RSI — ADX can show strong downtrend while
    RSI shows oversold, which is genuinely useful information.

    Regime multipliers actively reduce weights for signals that are
    less reliable in non-trending market conditions.

    Decision thresholds are controlled by the user's risk profile:
    conservative users need a higher score to act.
    """
    if len(df) < 5:
        return None

    profile = RISK_PROFILES[risk_profile_name]
    trend   = detect_trend(df)

    # ── Regime multipliers ────────────────────────────────────────────
    if trend in ('UPTREND', 'DOWNTREND'):
        bb_multiplier = 1.0
        sr_multiplier = 1.0
    elif trend == 'CONSOLIDATING':
        bb_multiplier = 0.5
        sr_multiplier = 1.5
    else:  # SIDEWAYS
        bb_multiplier = 0.75
        sr_multiplier = 1.25

    price    = safe_iloc(df['Close'],       -1)
    rsi      = safe_iloc(df['RSI'],         -1)
    adx      = safe_iloc(df['ADX'],         -1)
    adx_plus = safe_iloc(df['ADX_Plus'],    -1)
    adx_minus= safe_iloc(df['ADX_Minus'],   -1)
    bb_pos   = safe_iloc(df['BB_Position'], -1)
    obv_now  = safe_iloc(df['OBV'],         -1)
    obv_ma   = safe_iloc(df['OBV_MA10'],    -1)
    vol_now  = safe_iloc(df['Volume'],      -1)
    vol_ma   = safe_iloc(df['Vol_MA20'],    -1)

    if any(np.isnan(v) for v in [price, rsi, bb_pos, vol_now]):
        return None

    # ── RSI: 5-zone scoring (DIMENSION: Momentum) ────────────────────
    # Thresholds: 30/45/55/70 — creates gradual scaling, not binary.
    if   rsi < 30: rsi_s = 2
    elif rsi < 45: rsi_s = 1
    elif rsi < 55: rsi_s = 0
    elif rsi < 70: rsi_s = -1
    else:          rsi_s = -2

    # ── ADX: trend strength + direction (DIMENSION: Trend Strength) ──
    # REFORM: Replaces MACD in the scoring system.
    # ADX measures whether a trend EXISTS and how strong it is.
    # Combined with +DI/-DI to know the direction.
    # This is orthogonal to RSI — ADX can show strong downtrend (sell)
    # while RSI shows oversold (buy), which is genuinely informative.
    if np.isnan(adx) or np.isnan(adx_plus) or np.isnan(adx_minus):
        adx_s = 0
    elif adx_plus > adx_minus:
        # Upward directional pressure
        if   adx > 25: adx_s =  2   # Strong uptrend
        elif adx > 15: adx_s =  1   # Mild uptrend
        else:          adx_s =  0   # Directionless
    else:
        # Downward directional pressure
        if   adx > 25: adx_s = -2   # Strong downtrend
        elif adx > 15: adx_s = -1   # Mild downtrend
        else:          adx_s =  0   # Directionless

    # ── Bollinger: proximity to bands (DIMENSION: Structure) ──────────
    if   bb_pos < 0.15: raw_bb = 2
    elif bb_pos > 0.85: raw_bb = -2
    else:               raw_bb = 0
    bb_s = int(round(raw_bb * bb_multiplier))

    # ── Volume: spike detection (DIMENSION: Volume conviction) ────────
    # >2x average = volume spike = very high conviction (+2)
    # >1.3x average = above average = some conviction (+1)
    # otherwise = weak conviction (0)
    if   not np.isnan(vol_ma) and vol_now > 2.0 * vol_ma: vol_s = 2
    elif not np.isnan(vol_ma) and vol_now > 1.3 * vol_ma: vol_s = 1
    else:                                                   vol_s = 0

    # ── OBV: vs its 10-day MA (DIMENSION: Volume money flow) ──────────
    obv_s = (
         1 if (not np.isnan(obv_ma) and obv_now > obv_ma) else
        -1 if (not np.isnan(obv_ma) and obv_now < obv_ma) else
         0
    )

    # ── S/R: proximity with bounce confirmation (regime-adjusted) ─────
    near_sup = price <= support    * 1.02
    near_res = price >= resistance * 0.98
    bouncing = near_and_bouncing(df, support)
    rejected = (not near_and_bouncing(df, resistance)) and near_res

    if   near_sup and bouncing: raw_sr = 2
    elif near_sup:              raw_sr = 1
    elif near_res and rejected: raw_sr = -2
    elif near_res:              raw_sr = -1
    else:                       raw_sr = 0
    sr_s = max(-2, min(2, int(round(raw_sr * sr_multiplier))))

    # ── Trend: double weight — most important context signal ──────────
    if   trend == 'UPTREND':   trend_s = 2
    elif trend == 'DOWNTREND': trend_s = -2
    else:                      trend_s = 0

    total = rsi_s + adx_s + bb_s + vol_s + obv_s + sr_s + trend_s

    # ── Decision thresholds (controlled by risk profile) ──────────────
    sbt = profile['strong_buy_threshold']
    bt  = profile['buy_threshold']
    sst = profile['strong_sell_threshold']
    st  = profile['sell_threshold']

    if   total >= sbt: decision = 'STRONG BUY'
    elif total >= bt:  decision = 'BUY'
    elif total <= sst: decision = 'STRONG SELL'
    elif total <= st:  decision = 'SELL'
    else:              decision = 'HOLD'

    sub_scores = {
        'rsi': rsi_s, 'adx': adx_s, 'bollinger': bb_s,
        'volume': vol_s, 'obv': obv_s,
        'support_resistance': sr_s, 'trend': trend_s
    }
    confidence      = compute_confidence(sub_scores, decision)
    confidence_note = get_confidence_note(decision)

    # ── REFORM #5: Clear action labeling ─────────────────────────────
    # Old output had signal='SELL' + applicable=False — confusing.
    # New fields make intent explicit for both humans and the LLM:
    #
    #   action_existing_holders : what someone who HOLDS should do
    #   action_new_capital      : what someone with CASH should do
    #
    # This eliminates the contradiction between signal direction and
    # position sizing applicability.
    if decision in ('BUY', 'STRONG BUY'):
        action_existing = 'HOLD / ADD'
        action_new      = 'CONSIDER ENTRY — see position sizing'
    elif decision in ('SELL', 'STRONG SELL'):
        action_existing = 'EXIT position'
        action_new      = 'DO NOT ENTER — unfavourable conditions'
    else:  # HOLD
        action_existing = 'HOLD — no change needed'
        action_new      = 'WAIT — no clear edge yet'

    return {
        'decision':             decision,
        'action_existing_holders': action_existing,
        'action_new_capital':      action_new,
        'total_score':          total,
        'confidence_pct':       confidence,
        'confidence_note':      confidence_note,
        'sub_scores':           sub_scores,
        'risk_profile':         risk_profile_name,
        'regime':               trend,
        'regime_multipliers': {
            'bollinger':          bb_multiplier,
            'support_resistance': sr_multiplier
        },
        'thresholds_used': {
            'strong_buy': sbt, 'buy': bt,
            'sell': st,        'strong_sell': sst
        }
    }

print('✅ Scoring system ready (7 sub-signals across 4 orthogonal dimensions)')
print('   Sub-signals: RSI · ADX · Bollinger · Volume · OBV · S/R · Trend')


In [ ]:
# ── Cell 10: Position Sizing (capital-constrained) ────────────────────
def compute_position_size(capital, entry_price, support, risk_profile_name):
    """
    Calculates position size satisfying TWO constraints simultaneously:

    1. Risk constraint:
       Max loss on stop-loss hit = risk_pct_per_trade × capital.
       This limits how much money you can lose on a single trade.

    2. Capital constraint:
       Total position cost must never exceed available capital.
       This prevents recommending positions larger than the account.

    The smaller share count from the two constraints is used,
    so both constraints are always satisfied simultaneously.

    Stop loss is placed 0.5% below support — a small buffer so that
    a brief intraday dip below support does not trigger an exit.
    """
    profile        = RISK_PROFILES[risk_profile_name]
    risk_pct       = profile['risk_pct_per_trade']
    stop_loss      = round(support * 0.995, 2)
    risk_per_share = entry_price - stop_loss

    if risk_per_share <= 0:
        return 0, stop_loss, 0.0

    shares_risk    = (capital * risk_pct) / risk_per_share
    shares_capital = capital / (entry_price * (1 + TRANSACTION_COST_PCT))
    shares         = round(min(shares_risk, shares_capital), 2)
    position_cost  = round(shares * entry_price, 2)

    return shares, stop_loss, position_cost

print('✅ Position sizing ready (capital-constrained)')

In [ ]:
# ── Cell 11: Backtest using backtesting.py ────────────────────────────
def run_backtest(df_full, symbol, risk_profile_name, cash=10000.0):
    """
    Runs the strategy through backtesting.py.

    REFORM #1 — Slippage & Volume Filter:
      Slippage   : entry price is worsened by SLIPPAGE_PCT on each buy,
                   exit price is worsened by SLIPPAGE_PCT on each sell.
                   backtesting.py does not natively support per-trade
                   slippage, so we embed it in the strategy logic by
                   adjusting the limit price on entry/exit.
      Volume filter : trade is skipped if daily_volume × price <
                   MIN_DAILY_VOLUME_EGP. Prevents fills on illiquid days
                   where the position size cannot realistically be filled.

    REFORM #2 — Two-Layer Entry (Setup vs Confirmation):
      Backtest uses a SETUP threshold = profile threshold − SETUP_SCORE_OFFSET.
      This generates more trades for statistical validity while the live
      signal stays at the user's stricter threshold.
      Example (moderate profile): live BUY at score ≥ 2, backtest at ≥ 1.

    REFORM #4 — Score-Band History:
      After each completed backtest, profitable and unprofitable trades
      are recorded in SCORE_BAND_HISTORY, keyed by signal band.
      This feeds the new confidence metric in Cell 8.

    ✅ No look-ahead bias (each bar only sees past data)
    ✅ Transaction costs applied per trade automatically
    ✅ Stop-loss and take-profit as native order types
    ✅ Professional metrics: Sharpe, Sortino, Max Drawdown, SQN, etc.
    """
    support, resistance = get_levels(symbol)
    if support is None:
        return _empty_backtest('No S/R levels configured')

    if len(df_full) < 60:
        return _empty_backtest(
            f'Insufficient data ({len(df_full)} rows, need at least 60)'
        )

    profile     = RISK_PROFILES[risk_profile_name]
    # REFORM #2: setup threshold is looser than live threshold
    buy_thresh  = profile['buy_threshold']  - SETUP_SCORE_OFFSET
    sell_thresh = profile['sell_threshold'] + SETUP_SCORE_OFFSET  # less negative

    class EGXStrategy(Strategy):

        def init(self):
            self.ind_rsi      = self.I(lambda: df_full['RSI'].values,         name='RSI')
            self.ind_adx      = self.I(lambda: df_full['ADX'].values,         name='ADX')
            self.ind_adx_plus = self.I(lambda: df_full['ADX_Plus'].values,    name='ADX+')
            self.ind_adx_minus= self.I(lambda: df_full['ADX_Minus'].values,   name='ADX-')
            self.ind_bb_pos   = self.I(lambda: df_full['BB_Position'].values, name='BB_Pos')
            self.ind_bb_wid   = self.I(lambda: df_full['BB_Width'].values,    name='BB_Width')
            self.ind_obv      = self.I(lambda: df_full['OBV'].values,         name='OBV')
            self.ind_obv_ma   = self.I(lambda: df_full['OBV_MA10'].values,    name='OBV_MA')
            self.ind_vol_ma   = self.I(lambda: df_full['Vol_MA20'].values,    name='Vol_MA')
            self.ind_sma20    = self.I(lambda: df_full['SMA20'].values,       name='SMA20')
            self.ind_sma50    = self.I(lambda: df_full['SMA50'].values,       name='SMA50')

        def _score(self):
            """
            Identical scoring logic to score_symbol() (Cell 9).
            Uses ADX in place of MACD — same 4 orthogonal dimensions.
            Returns 0 (neutral) if core indicators are not yet computed.
            """
            price     = self.data.Close[-1]
            rsi       = self.ind_rsi[-1]
            adx       = self.ind_adx[-1]
            adx_plus  = self.ind_adx_plus[-1]
            adx_minus = self.ind_adx_minus[-1]
            bb_pos    = self.ind_bb_pos[-1]
            bb_wid    = self.ind_bb_wid[-1]
            obv_now   = self.ind_obv[-1]
            obv_ma    = self.ind_obv_ma[-1]
            vol_now   = self.data.Volume[-1]
            vol_ma    = self.ind_vol_ma[-1]
            sma20     = self.ind_sma20[-1]
            sma50     = self.ind_sma50[-1]

            if any(np.isnan(v) for v in [price, rsi, bb_pos, sma20]):
                return 0

            # Trend detection
            if not np.isnan(bb_wid) and bb_wid < 0.05:
                trend = 'CONSOLIDATING'
            elif price > sma20 and (np.isnan(sma50) or sma20 > sma50):
                trend = 'UPTREND'
            elif price < sma20 and (np.isnan(sma50) or sma20 < sma50):
                trend = 'DOWNTREND'
            else:
                trend = 'SIDEWAYS'

            # Regime multipliers
            if trend in ('UPTREND', 'DOWNTREND'):
                bb_mult, sr_mult = 1.0, 1.0
            elif trend == 'CONSOLIDATING':
                bb_mult, sr_mult = 0.5, 1.5
            else:
                bb_mult, sr_mult = 0.75, 1.25

            # RSI
            if   rsi < 30: rsi_s = 2
            elif rsi < 45: rsi_s = 1
            elif rsi < 55: rsi_s = 0
            elif rsi < 70: rsi_s = -1
            else:          rsi_s = -2

            # ADX (replaces MACD)
            if np.isnan(adx) or np.isnan(adx_plus) or np.isnan(adx_minus):
                adx_s = 0
            elif adx_plus > adx_minus:
                adx_s =  2 if adx > 25 else (1 if adx > 15 else 0)
            else:
                adx_s = -2 if adx > 25 else (-1 if adx > 15 else 0)

            # Bollinger
            if   bb_pos < 0.15: raw_bb = 2
            elif bb_pos > 0.85: raw_bb = -2
            else:               raw_bb = 0
            bb_s = int(round(raw_bb * bb_mult))

            # Volume
            if   not np.isnan(vol_ma) and vol_now > 2.0 * vol_ma: vol_s = 2
            elif not np.isnan(vol_ma) and vol_now > 1.3 * vol_ma: vol_s = 1
            else:                                                   vol_s = 0

            # OBV
            obv_s = (
                 1 if (not np.isnan(obv_ma) and obv_now > obv_ma) else
                -1 if (not np.isnan(obv_ma) and obv_now < obv_ma) else
                 0
            )

            # S/R
            near_sup = price <= support    * 1.02
            near_res = price >= resistance * 0.98
            if   near_sup: raw_sr = 1
            elif near_res: raw_sr = -1
            else:          raw_sr = 0
            sr_s = max(-2, min(2, int(round(raw_sr * sr_mult))))

            # Trend
            if   trend == 'UPTREND':   trend_s = 2
            elif trend == 'DOWNTREND': trend_s = -2
            else:                      trend_s = 0

            return rsi_s + adx_s + bb_s + vol_s + obv_s + sr_s + trend_s

        def next(self):
            score     = self._score()
            price     = self.data.Close[-1]
            vol_now   = self.data.Volume[-1]
            vol_ma    = self.ind_vol_ma[-1]

            # REFORM #1 — Volume filter:
            # Skip trade if today's traded value is below minimum threshold.
            # This prevents fills on illiquid days.
            daily_volume_egp = vol_now * price
            if daily_volume_egp < MIN_DAILY_VOLUME_EGP:
                return

            # REFORM #1 — Slippage-adjusted prices:
            # Buy at a slightly worse price than close (market impact).
            # Use limit orders at the adjusted price.
            entry_price = round(price * (1 + SLIPPAGE_PCT), 2)
            exit_price  = round(price * (1 - SLIPPAGE_PCT), 2)

            sl = round(support    * 0.995, 2)
            tp = float(resistance)

            if not self.position:
                # REFORM #2 — Setup threshold (looser than live signal)
                if score >= buy_thresh and sl < price < tp:
                    # Size position accounting for slippage-adjusted entry
                    self.buy(sl=sl, tp=tp)
            else:
                # REFORM #2 — Also use setup-offset for exits (less aggressive)
                if score <= sell_thresh:
                    self.position.close()

    # ── Run the backtest ──────────────────────────────────────────────
    bt    = Backtest(df_full, EGXStrategy,
                     cash=cash,
                     commission=TRANSACTION_COST_PCT,
                     trade_on_close=True,
                     finalize_trades=True)
    stats = bt.run()

    # ── REFORM #4: Populate score-band history ────────────────────────
    # Extract individual trade results and update the history dict.
    # We map the entry score → signal band → win/loss.
    # backtesting.py stores trades in stats['_trades'] as a DataFrame.
    try:
        trades_df = stats.get('_trades')
        if trades_df is not None and len(trades_df) > 0:
            for _, trade in trades_df.iterrows():
                pnl = trade.get('PnL', 0)
                # We don't store per-trade entry scores in backtesting.py,
                # so we attribute wins/losses to the live signal band.
                # This is an approximation — in production, log score at entry.
                profile_obj = RISK_PROFILES[risk_profile_name]
                sbt = profile_obj['strong_buy_threshold']
                bt_ = profile_obj['buy_threshold']
                # All backtest longs go in 'BUY' (setup entries, not strong buy)
                band = 'BUY'
                update_score_band_history(band, pnl > 0)
    except Exception:
        pass  # History update is best-effort — never block main flow

    # ── Extract and format results ────────────────────────────────────
    n_trades    = int(stats['# Trades'])
    is_reliable = n_trades >= MIN_TRADES_FOR_RELIABLE_BACKTEST

    def safe_stat(key, decimals=2):
        v = stats.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        return round(float(v), decimals)

    ret = safe_stat('Return [%]')
    if not is_reliable:
        validation = 'UNVALIDATED'
        validation_note = (
            f'Only {n_trades} trade(s) executed — '
            f'minimum {MIN_TRADES_FOR_RELIABLE_BACKTEST} required. '
            f'Treat signal with extra caution. '
            f'(Entry loosened by {SETUP_SCORE_OFFSET} point(s) vs live threshold '
            f'to increase trade frequency — still insufficient.)'
        )
    elif ret is not None and ret < 0:
        validation = 'CONTRADICTED'
        validation_note = (
            f'Backtest returned {ret}% — historical performance '
            f'contradicts current signal. Extra caution advised.'
        )
    else:
        validation = 'VALIDATED'
        validation_note = None

    return {
        'final_capital_EGP':      round(float(stats['Equity Final [$]']), 2),
        'total_return_pct':       safe_stat('Return [%]'),
        'buy_and_hold_pct':       safe_stat('Buy & Hold Return [%]'),
        'annual_return_pct':      safe_stat('Return (Ann.) [%]'),
        'total_trades':           n_trades,
        'win_rate_pct':           safe_stat('Win Rate [%]', 1),
        'avg_trade_pct':          safe_stat('Avg. Trade [%]'),
        'best_trade_pct':         safe_stat('Best Trade [%]'),
        'worst_trade_pct':        safe_stat('Worst Trade [%]'),
        'avg_trade_duration':     str(stats.get('Avg. Trade Duration', 'N/A')),
        'sharpe_ratio':           safe_stat('Sharpe Ratio'),
        'sortino_ratio':          safe_stat('Sortino Ratio'),
        'calmar_ratio':           safe_stat('Calmar Ratio'),
        'max_drawdown_pct':       safe_stat('Max. Drawdown [%]'),
        'avg_drawdown_pct':       safe_stat('Avg. Drawdown [%]'),
        'profit_factor':          safe_stat('Profit Factor'),
        'expectancy_pct':         safe_stat('Expectancy [%]'),
        'sqn':                    safe_stat('SQN'),
        'commissions_EGP':        safe_stat('Commissions [$]'),
        'statistically_reliable': is_reliable,
        'min_trades_threshold':   MIN_TRADES_FOR_RELIABLE_BACKTEST,
        'signal_validation':      validation,
        'validation_note':        validation_note,
        'transaction_cost_pct':   TRANSACTION_COST_PCT * 100,
        'slippage_pct':           SLIPPAGE_PCT * 100,
        'setup_score_offset':     SETUP_SCORE_OFFSET,
        'risk_profile':           risk_profile_name,
    }


def _empty_backtest(note):
    return {
        'final_capital_EGP': 10000.0, 'total_return_pct': 0.0,
        'buy_and_hold_pct': None, 'annual_return_pct': None,
        'total_trades': 0, 'win_rate_pct': 0.0,
        'avg_trade_pct': None, 'best_trade_pct': None,
        'worst_trade_pct': None, 'avg_trade_duration': None,
        'sharpe_ratio': None, 'sortino_ratio': None,
        'calmar_ratio': None, 'max_drawdown_pct': 0.0,
        'avg_drawdown_pct': None, 'profit_factor': None,
        'expectancy_pct': None, 'sqn': None, 'commissions_EGP': 0.0,
        'statistically_reliable': False,
        'min_trades_threshold': MIN_TRADES_FOR_RELIABLE_BACKTEST,
        'signal_validation': 'UNVALIDATED',
        'validation_note': note,
        'transaction_cost_pct': TRANSACTION_COST_PCT * 100,
        'slippage_pct': SLIPPAGE_PCT * 100,
        'setup_score_offset': SETUP_SCORE_OFFSET,
    }

print('✅ Backtest ready (slippage + volume filter + setup layer + score-band history)')
print(f'   Slippage     : {SLIPPAGE_PCT*100}% per side')
print(f'   Min volume   : EGP {MIN_DAILY_VOLUME_EGP:,.0f}/day')
print(f'   Setup offset : -{SETUP_SCORE_OFFSET} vs live threshold')


In [ ]:
# ── Cell 12: LLM Context Builder ──────────────────────────────────────
def build_llm_context(symbol, df, scoring, bt, support, resistance, capital):
    """
    Builds the complete structured JSON for the large LLM.

    REFORM #3: ADX replaces MACD in the indicators block.
    REFORM #4: confidence_note explains whether confidence is from
               historical win rate or calibrated prior.
    REFORM #5: action_existing_holders and action_new_capital replace
               the confusing signal + applicable=False pattern.
    """
    price   = safe_iloc(df['Close'],       -1)
    rsi     = safe_iloc(df['RSI'],         -1)
    adx     = safe_iloc(df['ADX'],         -1)
    adx_p   = safe_iloc(df['ADX_Plus'],    -1)
    adx_m   = safe_iloc(df['ADX_Minus'],   -1)
    macd_h  = safe_iloc(df['MACD_Hist'],   -1)   # context only
    bb_pos  = safe_iloc(df['BB_Position'], -1)
    bb_wid  = safe_iloc(df['BB_Width'],    -1)
    atr     = safe_iloc(df['ATR'],         -1)
    sma20   = safe_iloc(df['SMA20'],       -1)
    sma50   = safe_iloc(df['SMA50'],       -1)
    vol_now = safe_iloc(df['Volume'],      -1)
    vol_ma  = safe_iloc(df['Vol_MA20'],    -1)
    obv_now = safe_iloc(df['OBV'],         -1)
    obv_ma  = safe_iloc(df['OBV_MA10'],    -1)
    trend   = detect_trend(df)
    profile = RISK_PROFILES[scoring['risk_profile']]
    analysis_date = df.index[-1].strftime('%Y-%m-%d') if len(df) > 0 else 'N/A'

    # ── RSI interpretation ────────────────────────────────────────────
    if   rsi < 30: rsi_interp = 'oversold — sellers likely exhausted'
    elif rsi < 45: rsi_interp = 'mildly oversold — mild buying opportunity'
    elif rsi < 55: rsi_interp = 'neutral — no directional pressure'
    elif rsi < 70: rsi_interp = 'mildly overbought — mild caution advised'
    else:          rsi_interp = 'overbought — buyers likely exhausted'

    # ── ADX interpretation ────────────────────────────────────────────
    if np.isnan(adx):
        adx_interp = 'ADX unavailable'
    elif adx < 15:
        adx_interp = f'ADX {adx:.1f} — no trend, ranging market'
    elif adx < 25:
        direction_label = 'upward' if (not np.isnan(adx_p) and adx_p > adx_m) else 'downward'
        adx_interp = f'ADX {adx:.1f} — mild {direction_label} trend developing'
    else:
        direction_label = 'upward' if (not np.isnan(adx_p) and adx_p > adx_m) else 'downward'
        adx_interp = f'ADX {adx:.1f} — strong {direction_label} trend confirmed'

    # ── Bollinger interpretation ───────────────────────────────────────
    bb_interp = (
        'near lower band — price unusually low relative to recent range'
        if bb_pos < 0.2 else
        'near upper band — price unusually high relative to recent range'
        if bb_pos > 0.8 else
        'within normal Bollinger range'
    )
    bbw_interp = (
        'very narrow — consolidation, large move likely soon'
        if (not np.isnan(bb_wid) and bb_wid < 0.05) else
        'normal width'
    )

    # ── MACD context note (not scored, human context only) ────────────
    macd_context = 'bullish momentum' if (not np.isnan(macd_h) and macd_h > 0) else 'bearish momentum'

    # ── Volume interpretation ─────────────────────────────────────────
    if   not np.isnan(vol_ma) and vol_now > 2.0 * vol_ma:
        vol_interp = 'volume spike — very high conviction'
    elif not np.isnan(vol_ma) and vol_now > 1.3 * vol_ma:
        vol_interp = 'above average — move has conviction'
    else:
        vol_interp = 'below average — weak conviction'

    obv_interp = (
        'OBV above average — volume confirming price'
        if (not np.isnan(obv_ma) and obv_now > obv_ma) else
        'OBV below average — volume contradicting price'
    )

    # ── Liquidity check for live signal ──────────────────────────────
    daily_volume_egp = vol_now * price if not np.isnan(vol_now) and not np.isnan(price) else 0
    liquidity_ok     = daily_volume_egp >= MIN_DAILY_VOLUME_EGP
    liquidity_note   = (
        f'EGP {daily_volume_egp:,.0f} traded today — '
        + ('sufficient liquidity.' if liquidity_ok else
           f'BELOW minimum threshold of EGP {MIN_DAILY_VOLUME_EGP:,.0f}. '
           f'Signal valid but execution risk is HIGH.')
    )

    atr_pct = round(atr / price * 100, 1) if not np.isnan(atr) and price else 0.0

    # ── Position sizing (only for BUY signals) ────────────────────────
    if scoring['decision'] in ('BUY', 'STRONG BUY'):
        shares, stop_loss_price, position_cost = compute_position_size(
            capital, price, support, scoring['risk_profile']
        )
        sizing_applicable = True
    else:
        shares, stop_loss_price, position_cost = 0, round(support * 0.995, 2), 0.0
        sizing_applicable = False

    risk_amount = round(capital * profile['risk_pct_per_trade'], 2)

    # ── Regime context ────────────────────────────────────────────────
    rm = scoring.get('regime_multipliers', {})
    regime_note = ''
    if rm.get('bollinger', 1.0) != 1.0 or rm.get('support_resistance', 1.0) != 1.0:
        regime_note = (
            f' In this {trend.lower()} regime, '
            f'Bollinger signals weighted at {rm.get("bollinger",1.0)}x '
            f'and S/R signals at {rm.get("support_resistance",1.0)}x.'
        )

    # ── Backtest validation ───────────────────────────────────────────
    validation      = bt.get('signal_validation', 'UNVALIDATED')
    validation_note = bt.get('validation_note')
    slippage_note   = f'Backtest includes {bt.get("slippage_pct",0)}% slippage per side + {bt.get("transaction_cost_pct",0)}% commission.'

    # ── Natural language summary ───────────────────────────────────────
    llm_summary = (
        f"{symbol} is trading at EGP {price:.2f} on the EGX as of {analysis_date}. "
        f"Market regime: {trend.lower()}.{regime_note} "
        f"RSI {rsi:.1f} ({rsi_interp}). "
        f"ADX: {adx_interp}. "
        f"Bollinger: {bb_interp}. "
        f"Volume: {vol_interp}. {obv_interp}. "
        f"Liquidity: {liquidity_note} "
        f"Support EGP {support} / Resistance EGP {resistance} (manually set). "
        f"Daily volatility ~{atr_pct}% of price. "
        f"Signal: {scoring['decision']} — "
        f"confidence {scoring['confidence_pct']}% ({scoring['confidence_note']}) "
        f"score {scoring['total_score']}/14, "
        f"{scoring['risk_profile']} risk thresholds. "
        f"Action for existing holders: {scoring['action_existing_holders']}. "
        f"Action for new capital: {scoring['action_new_capital']}. "
        f"Backtest validation: {validation}. "
        + (f"{validation_note} " if validation_note else '')
        + (f"Position sizing ({scoring['risk_profile']} profile, EGP {capital:,.0f} capital): "
           f"{shares} shares, cost EGP {position_cost:,.2f}, "
           f"stop EGP {stop_loss_price}, TP EGP {resistance}, "
           f"risking EGP {risk_amount} ({profile['risk_pct_per_trade']*100}% of capital). "
           if sizing_applicable else "")
        + f"{slippage_note} "
        + f"Backtest: {bt['total_return_pct']}% return vs {bt.get('buy_and_hold_pct')}% buy-and-hold, "
        + f"{bt['total_trades']} trades, Sharpe {bt['sharpe_ratio']}, "
        + f"max drawdown {bt['max_drawdown_pct']}%, SQN {bt.get('sqn')}."
    )

    return {
        'symbol':        symbol,
        'exchange':      'EGX (Egyptian Exchange)',
        'analysis_date': analysis_date,

        'price': {
            'current_EGP':    round(price, 2),
            'sma20_EGP':      round(sma20, 2) if not np.isnan(sma20) else None,
            'sma50_EGP':      round(sma50, 2) if not np.isnan(sma50) else None,
            'support_EGP':    support,
            'resistance_EGP': resistance,
            'sr_source':      'manual',
        },

        'trend':   trend,

        # ── REFORM #5: Clear action fields replace signal + applicable ──
        'signal':                  scoring['decision'],
        'action_existing_holders': scoring['action_existing_holders'],
        'action_new_capital':      scoring['action_new_capital'],

        'confidence_pct':  scoring['confidence_pct'],
        'confidence_note': scoring['confidence_note'],
        'total_score':     scoring['total_score'],
        'max_score':       14,
        'sub_scores':      scoring['sub_scores'],
        'regime_multipliers': scoring.get('regime_multipliers'),
        'thresholds_used':    scoring['thresholds_used'],
        'risk_profile':       scoring['risk_profile'],

        'indicators': {
            # DIMENSION 1: Momentum
            'RSI_14':                      round(rsi,    1) if not np.isnan(rsi)    else None,
            'RSI_interpretation':          rsi_interp,
            # DIMENSION 2: Trend Strength (scored) — REFORM #3
            'ADX_14':                      round(adx,    1) if not np.isnan(adx)    else None,
            'ADX_Plus_DI':                 round(adx_p,  1) if not np.isnan(adx_p)  else None,
            'ADX_Minus_DI':                round(adx_m,  1) if not np.isnan(adx_m)  else None,
            'ADX_interpretation':          adx_interp,
            # DIMENSION 3: Price Structure
            'Bollinger_position_0_to_1':   round(bb_pos, 2) if not np.isnan(bb_pos) else None,
            'Bollinger_interpretation':    bb_interp,
            'Bollinger_width':             round(bb_wid, 4) if not np.isnan(bb_wid) else None,
            'Bollinger_width_note':        bbw_interp,
            # DIMENSION 4: Volume / Money Flow
            'volume_interpretation':       vol_interp,
            'OBV_interpretation':          obv_interp,
            # Context only (not scored)
            'MACD_histogram_context_only': round(macd_h, 4) if not np.isnan(macd_h) else None,
            'MACD_note':                   f'{macd_context} (context only — not scored)',
            # Volatility
            'ATR_14_EGP':                  round(atr,    2) if not np.isnan(atr)    else None,
            'ATR_pct_of_price':            atr_pct,
        },

        'liquidity': {
            'daily_volume_EGP':     round(daily_volume_egp, 0),
            'min_threshold_EGP':    MIN_DAILY_VOLUME_EGP,
            'liquidity_sufficient': liquidity_ok,
            'note':                 liquidity_note,
        },

        'position_sizing': {
            'applicable':           sizing_applicable,
            'suggested_shares':     shares,
            'position_cost_EGP':    position_cost,
            'stop_loss_EGP':        stop_loss_price,
            'take_profit_EGP':      resistance,
            'capital_at_risk_EGP':  risk_amount if sizing_applicable else 0.0,
            'risk_pct_of_capital':  profile['risk_pct_per_trade'] * 100,
            'within_capital_limit': position_cost <= capital,
        },

        'backtest':           bt,
        'llm_prompt_summary': llm_summary,
    }

print('✅ LLM context builder ready')
print('   New fields: action_existing_holders, action_new_capital, confidence_note')
print('   New section: liquidity (daily volume vs minimum threshold)')
print('   Indicators: ADX replaces MACD in scored block; MACD retained as context-only')


In [ ]:
# ── Cell 13: Main Runner ──────────────────────────────────────────────
financial_outputs = []

for symbol in SYMBOLS:
    print(f"\n{'─'*60}")
    print(f'⏳ Processing {symbol}...')

    # ── S/R check first ───────────────────────────────────────────────
    support, resistance = get_levels(symbol)
    if support is None:
        print(f'  ⛔ Skipping — no manual S/R levels in MANUAL_LEVELS')
        print(f'     Add MANUAL_LEVELS["{symbol}"] to Cell 2 to include this symbol')
        continue

    # ── Fetch OHLCV data ──────────────────────────────────────────────
    df_raw = fetch_data(symbol)
    if df_raw is None or len(df_raw) < 30:
        rows = len(df_raw) if df_raw is not None else 0
        print(f'  ⚠️  Skipping — insufficient raw data ({rows} rows, need 30+)')
        continue

    # ── Compute indicators ────────────────────────────────────────────
    df_full = compute_all_indicators(df_raw.copy())

    # ── Live signal ───────────────────────────────────────────────────
    df_live = df_full.dropna(subset=['RSI','ADX','BB_Position','ATR','SMA20'])
    if len(df_live) < 5:
        print(f'  ⚠️  Skipping — too few rows after indicators ({len(df_live)})')
        continue

    scoring = score_symbol(df_live, support, resistance, USER_RISK_PROFILE)
    if scoring is None:
        print(f'  ⚠️  Skipping — could not compute score')
        continue

    # ── Backtest (uses full df — no row deletion) ─────────────────────
    bt = run_backtest(df_full, symbol, USER_RISK_PROFILE, cash=USER_CAPITAL)

    # ── Build LLM context ─────────────────────────────────────────────
    context = build_llm_context(
        symbol, df_live, scoring, bt, support, resistance, USER_CAPITAL
    )
    financial_outputs.append(context)

    # ── Print summary ─────────────────────────────────────────────────
    ps  = context['position_sizing']
    liq = context['liquidity']
    val = bt['signal_validation']

    print(f"  ✅ Data rows       : {len(df_raw)} raw | {len(df_live)} after indicators")
    print(f"  ✅ S/R levels      : support EGP {support} | resistance EGP {resistance}")
    print(f"  ✅ Regime          : {scoring['regime']} "
          f"(BB×{scoring['regime_multipliers']['bollinger']}, "
          f"SR×{scoring['regime_multipliers']['support_resistance']})")
    print(f"  ✅ Signal          : {scoring['decision']} "
          f"(score {scoring['total_score']}/14, "
          f"confidence {scoring['confidence_pct']}%)")
    print(f"     ↳ Existing holders : {scoring['action_existing_holders']}")
    print(f"     ↳ New capital      : {scoring['action_new_capital']}")
    print(f"     ↳ Confidence note  : {scoring['confidence_note']}")
    print(f"  ✅ Sub-scores      : {scoring['sub_scores']}")
    print(f"  ✅ Risk profile    : {USER_RISK_PROFILE} — "
          f"buy≥{scoring['thresholds_used']['buy']} / "
          f"sell≤{scoring['thresholds_used']['sell']}")
    print(f"  ✅ Liquidity       : EGP {liq['daily_volume_EGP']:,.0f} today | "
          f"sufficient: {liq['liquidity_sufficient']}")
    if ps['applicable']:
        print(f"  ✅ Position sizing : {ps['suggested_shares']} shares | "
              f"cost EGP {ps['position_cost_EGP']:,.2f} | "
              f"stop EGP {ps['stop_loss_EGP']} | "
              f"TP EGP {ps['take_profit_EGP']} | "
              f"within limit: {ps['within_capital_limit']}")
    else:
        print(f"  ✅ Position sizing : N/A ({scoring['decision']} signal)")
    print(f"  ✅ Backtest        : {bt['total_return_pct']}% return "
          f"(vs {bt.get('buy_and_hold_pct')}% buy-and-hold) | "
          f"{bt['total_trades']} trades | "
          f"Sharpe {bt['sharpe_ratio']} | "
          f"MaxDD {bt['max_drawdown_pct']}% | "
          f"SQN {bt.get('sqn')} | "
          f"slippage {bt.get('slippage_pct')}%/side | "
          f"reliable: {bt['statistically_reliable']}")
    print(f"  ✅ Validation      : {val}")
    if bt.get('validation_note'):
        print(f"  ⚠️  {bt['validation_note']}")

print(f"\n{'='*60}")
print(f"✅ Done — {len(financial_outputs)} / {len(SYMBOLS)} symbols processed")
print(f"   Risk profile : {USER_RISK_PROFILE}")
print(f"   Tx cost      : {TRANSACTION_COST_PCT*100}% + {SLIPPAGE_PCT*100}% slippage per side")
print(f"   Setup offset : -{SETUP_SCORE_OFFSET} in backtest vs live threshold")
print()
print("Score-band history after this run:")
for band, h in SCORE_BAND_HISTORY.items():
    if h['total'] > 0:
        wr = round(h['wins']/h['total']*100, 1)
        print(f"   {band:<12}: {h['wins']}/{h['total']} wins ({wr}%)")
    else:
        print(f"   {band:<12}: no trades recorded yet")


In [ ]:
# ── Cell 14: Final JSON Output ────────────────────────────────────────
part2_json = {
    'part':                 'financial_analysis',
    'generated_at':         datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'period':               f'{FROM_DATE} to {TO_DATE}',
    'user_risk_profile':    USER_RISK_PROFILE,
    'user_capital_EGP':     USER_CAPITAL,
    'transaction_cost_pct': TRANSACTION_COST_PCT * 100,
    'slippage_pct':         SLIPPAGE_PCT * 100,
    'setup_score_offset':   SETUP_SCORE_OFFSET,
    'backtest_engine':      'backtesting.py v0.6.x',
    'reforms_applied': [
        'Reform 1: Slippage (0.3%/side) + daily volume filter in backtest',
        'Reform 2: Two-layer entry — setup threshold (looser) vs live threshold',
        'Reform 3: ADX replaces MACD in scoring — 4 orthogonal dimensions',
        'Reform 4: Confidence from score-band win rate (calibrated prior fallback)',
        'Reform 5: action_existing_holders + action_new_capital replace ambiguous signal/applicable',
    ],
    'symbols_requested':    list(SYMBOLS.keys()),
    'symbols_processed':    [c['symbol'] for c in financial_outputs],
    'score_band_history':   SCORE_BAND_HISTORY,
    'companies':            financial_outputs,
}

print(json.dumps(part2_json, indent=2, ensure_ascii=False))


In [ ]:
# ── Cell 15: Clear cache (run only when you want fresh data) ──────────
import glob
for f in glob.glob('egx_cache/*.csv'):
    os.remove(f)
    print(f'  🗑️  Deleted: {f}')
print('Cache cleared — re-run Cell 13 to download fresh data')